# 🚗 Cuaderno de Trabajo: Chatbot de Diagnóstico Vehicular con Machine Learning
Este cuaderno contiene el flujo completo para tu tesis de diagnóstico vehicular en Carabayllo, 2026. 
Aquí podrás:
1. Crear y estructurar el dataset de síntomas y fallas en español peruano.
2. Entrenar el clasificador de Machine Learning (Random Forest).
3. Evaluar el modelo.
4. Probar diagnósticos individuales.

### Paso 1: Importar Librerías y Cargar Datos
En esta celda cargamos las librerías necesarias y el dataset de síntomas.

In [ ]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import os

# Comprobamos si el dataset existe
if os.path.exists('dataset_sintomas.csv'):
    df = pd.read_csv('dataset_sintomas.csv', encoding='utf-8')
    print(f'¡Éxito! Dataset cargado con {len(df)} registros de diagnóstico.')
    display(df.head())
else:
    print('Error: Asegúrate de que el archivo dataset_sintomas.csv esté en la misma carpeta.')

### Paso 2: Entrenar el Modelo de Machine Learning
Utilizaremos **TF-IDF** para convertir el texto de los síntomas en vectores numéricos, y luego entrenaremos un clasificador **Random Forest**.

In [ ]:
X = df['sintoma']
y = df['falla']

# 1. Convertimos texto a vectores numéricos
vectorizador = TfidfVectorizer(lowercase=True, strip_accents='unicode')
X_vectorizado = vectorizador.fit_transform(X)

# 2. Entrenamos el clasificador
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_vectorizado, y)

# 3. Evaluamos la exactitud
predicciones = modelo.predict(X_vectorizado)
exactitud = accuracy_score(y, predicciones)
print(f'Exactitud del modelo: {exactitud * 100:.2f}%')

# 4. Guardamos el modelo para usarlo en el Chatbot de WhatsApp
joblib.dump(modelo, 'modelo_diagnostico.pkl')
joblib.dump(vectorizador, 'vectorizador_tfidf.pkl')
print('¡Archivos del modelo pkl guardados con éxito!')

### Paso 3: Probar el Diagnóstico Interactivo
Escribe un síntoma en la variable `mi_sintoma` para probar la predicción de la IA en tiempo real.

In [ ]:
# Cargar modelo guardado
modelo_cargado = joblib.load('modelo_diagnostico.pkl')
vec_cargado = joblib.load('vectorizador_tfidf.pkl')

# Puedes cambiar este síntoma para probar diferentes fallas:
mi_sintoma = 'siento que el carro chilla al frenar'

# Predicción
entrada_vec = vec_cargado.transform([mi_sintoma])
prediccion = modelo_cargado.predict(entrada_vec)[0]
probabilidades = modelo_cargado.predict_proba(entrada_vec)
confianza = max(probabilidades[0])

print('=' * 60)
print(f'Síntoma reportado: "{mi_sintoma}"')
print(f'Diagnóstico estimado por IA: {prediccion}')
print(f'Nivel de confianza: {confianza * 100:.2f}%')
print('=' * 60)